## BioPAX abstraction: Pathway-centered view

In [1]:
from SPARQLWrapper import SPARQLWrapper, JSON, CSV, N3, XML, TURTLE
import subprocess
import time
import os
from requests.utils import requote_uri
from urllib.parse import quote
import re
import rdflib
import pandas as pd
import math
import networkx as nx
import IPython

In [2]:
endpoint_reactome = "http://localhost:3030/reactome"
rdfFormat = "turtle"
current_directory = os.getcwd()
BioPAX_Ontology_file_path = os.path.join(current_directory, '../', 'BioPAXData', 'biopax-level3.owl')
ReactomeBioPAX_file_path = os.path.join(current_directory, '../', 'BioPAXData', 'Homo_sapiens_v94.owl')

In [3]:
def displaySparqlResults(results):
    '''
    Displays as HTML the result of a SPARQLWrapper query in a Jupyter notebook.
    
        Parameters:
            results (dictionnary): the result of a call to SPARQLWrapper.query().convert()
    '''
    variableNames = results['head']['vars']
    tableCode = '<table><tr><th>{}</th></tr><tr>{}</tr></table>'.format('</th><th>'.join(variableNames), '</tr><tr>'.join('<td>{}</td>'.format('</td><td>'.join([row[vName]['value'] if vName in row.keys() else "&nbsp;" for vName in variableNames]))for row in results["results"]["bindings"]))
    IPython.display.display(IPython.display.HTML(tableCode))

In [5]:
def extract_prefix_mappings(prefixes_string):
    """
    Extract prefix mappings from the SPARQL prefixes string.
    
    Parameters:
    prefixes_string (str): String containing PREFIX declarations
    
    Returns:
    dict: Mapping of full URIs to their prefixes
    """
    # Extract prefix declarations using regex
    prefix_pattern = re.compile(r'PREFIX\s+(\w+):\s*<([^>]+)>', re.IGNORECASE)
    return {uri: prefix for prefix, uri in prefix_pattern.findall(prefixes_string)}

def convert_to_prefixed_uri(uri_string, prefix_mappings):
    """
    Convert a full URI to prefixed format.
    
    Parameters:
    uri_string (str): Full URI string
    prefix_mappings (dict): Mapping of URIs to prefixes
    
    Returns:
    str: URI in prefixed format (e.g., 'reactome:Protein')
    """
    for uri_base, prefix in prefix_mappings.items():
        if uri_string.startswith(uri_base):
            local_part = uri_string[len(uri_base):]
            
            return f"{prefix}:{local_part}"
    return uri_string  # Return original if no prefix matches

def save_for_cytoscape(sparql, prefixes_string, output_file, format='csv', separator=','):
    """
    Save SPARQL CONSTRUCT results in a format compatible with Cytoscape,
    using prefix notation for URIs.
    
    Parameters:
    sparql (SPARQLWrapper): Configured SPARQLWrapper instance with query
    prefixes_string (str): String containing PREFIX declarations
    output_file (str): Path to save the output file
    format (str): Output format ('csv' or 'tsv')
    separator (str): Column separator (',' for CSV, '\t' for TSV)
    """
    # Extract prefix mappings
    prefix_mappings = extract_prefix_mappings(prefixes_string)
    
    # Get the results as an RDF graph
    sparql.setReturnFormat(TURTLE)
    results = sparql.queryAndConvert()
    
    # Create an RDFlib graph
    g = rdflib.Graph()
    if isinstance(results, bytes):
        g.parse(data=results.decode('utf-8'), format='turtle')
    else:
        g.parse(data=results, format='turtle')
    
    # Convert triples to a list of dictionaries with prefixed URIs
    triples_data = []
    for s, p, o in g:# Extraction of "Signaling by EGFR" (R-HSA-177929) pathway from Reactome BioPAX export v65
        # Convert each URI to prefixed format
        subject = convert_to_prefixed_uri(str(s), prefix_mappings)
        predicate = convert_to_prefixed_uri(str(p), prefix_mappings)
        object_ = convert_to_prefixed_uri(str(o), prefix_mappings)
        
        triples_data.append({
            'Source': subject,
            'Interaction': predicate,
            'Target': object_
        })
    
    # Convert to DataFrame for easy CSV/TSV export
    df = pd.DataFrame(triples_data)
    
    # Save to file
    if format == 'csv':
        df.to_csv(output_file, index=False, sep=',')
    else:  # tsv
        df.to_csv(output_file, index=False, sep='\t')
    
    print(f"Saved {len(triples_data)} interactions to {output_file}")
    return df

def preview_network_data(df, n=5):
    """
    Preview the network data before importing into Cytoscape.
    
    Parameters:
    df (pandas.DataFrame): DataFrame containing the network data
    n (int): Number of rows to preview
    """
    print(f"\nPreview of network data ({len(df)} total interactions):")
    print(f"\nFirst {n} interactions:")
    print(df.head(n))
    
    # Print some basic network statistics
    unique_nodes = set(df['Source'].unique()) | set(df['Target'].unique())
    print(f"\nNetwork statistics:")
    print(f"Number of unique nodes: {len(unique_nodes)}")
    print(f"Number of interactions: {len(df)}")
    print(f"Unique interaction types:")
    for interaction in sorted(df['Interaction'].unique()):
        print(f"  - {interaction}")

In [6]:
prefixes = f"""
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs:<http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX dc: <http://purl.org/dc/elements/1.1/>
PREFIX dcterms: <http://purl.org/dc/terms/>
PREFIX chebi: <http://purl.obolibrary.org/obo/chebi/>
PREFIX chebidb: <http://purl.obolibrary.org/obo/CHEBI_>
PREFIX chebirel: <http://purl.obolibrary.org/obo/CHEBI#>
PREFIX oboInOwl: <http://www.geneontology.org/formats/oboInOwl#>
PREFIX bp3: <http://www.biopax.org/release/biopax-level3.owl#>
PREFIX reactome: <http://www.reactome.org/biopax/40/48887#>
PREFIX abstraction:<http://abstraction/#>
"""

In [ ]:
command = [
    '/home/cbeust/Softwares/JenaFuseki/apache-jena-fuseki-4.9.0/fuseki-server',
    '--file', ReactomeBioPAX_file_path,
    '--file', BioPAX_Ontology_file_path,
    '/reactome']

process = subprocess.Popen(command)
time.sleep(60)

16:04:59 INFO  Server          :: Dataset: in-memory: load file: /home/cbeust/Projects/2025/BioPAXPathwayAbstraction/Scripts/../BioPAXData/Homo_sapiens_v94.owl
16:05:00 WARN  riot            :: [line: 66845, col: 48] {W137} Input is large. Switching off checking for illegal reuse of rdf:ID's.
16:05:22 INFO  Server          :: Dataset: in-memory: load file: /home/cbeust/Projects/2025/BioPAXPathwayAbstraction/Scripts/../BioPAXData/biopax-level3.owl
16:05:22 INFO  Server          :: Running in read-only mode for /reactome
16:05:22 INFO  Server          :: Apache Jena Fuseki 4.9.0
16:05:22 INFO  Config          :: FUSEKI_HOME=/home/cbeust/Softwares/JenaFuseki/apache-jena-fuseki-4.9.0
16:05:22 INFO  Config          :: FUSEKI_BASE=/home/cbeust/Projects/2025/BioPAXPathwayAbstraction/Scripts/run
16:05:22 INFO  Config          :: Shiro file: file:///home/cbeust/Projects/2025/BioPAXPathwayAbstraction/Scripts/run/shiro.ini
16:05:22 INFO  Server          :: Database: in-memory, with files loaded
1

16:06:35 INFO  Fuseki          :: [4] POST http://localhost:3030/reactome/sparql
16:06:35 INFO  Fuseki          :: [4] Query = PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> PREFIX rdfs:<http://www.w3.org/2000/01/rdf-schema#> PREFIX owl: <http://www.w3.org/2002/07/owl#> PREFIX xsd: <http://www.w3.org/2001/XMLSchema#> PREFIX dc: <http://purl.org/dc/elements/1.1/> PREFIX dcterms: <http://purl.org/dc/terms/> PREFIX chebi: <http://purl.obolibrary.org/obo/chebi/> PREFIX chebidb: <http://purl.obolibrary.org/obo/CHEBI_> PREFIX chebirel: <http://purl.obolibrary.org/obo/CHEBI#> PREFIX oboInOwl: <http://www.geneontology.org/formats/oboInOwl#> PREFIX bp3: <http://www.biopax.org/release/biopax-level3.owl#> PREFIX reactome: <http://www.reactome.org/biopax/93/48887#> PREFIX abstraction:<http://abstraction/#>  SELECT DISTINCT ?pathwayName ?nextName ?tag WHERE {   ?pathway rdf:type bp3:Pathway .   ?nextPathway rdf:type bp3:Pathway .      ?pathway bp3:pathwayOrder ?pathwayStep .   ?nextPathw

### SPARQL queries for abstraction

#### 1 - IsAChildOf

Defines a relation that decribes the direct hierarchy of pathways.

In [ ]:
start_time = time.time()

query_is_a_child_of = """
SELECT DISTINCT ?pathwayID ?subPathwayID
WHERE {
  ?pathway rdf:type bp3:Pathway .
  ?pathway bp3:pathwayComponent ?subPathway .
  ?subPathway rdf:type bp3:Pathway .

  ?pathway bp3:xref ?pathwayXref .
  ?pathwayXref rdf:type bp3:UnificationXref .
  ?pathwayXref bp3:db "Reactome" .
  ?pathwayXref bp3:id ?pathwayID .
  
  ?subPathway bp3:xref ?subPathwayXref .
  ?subPathwayXref rdf:type bp3:UnificationXref .
  ?subPathwayXref bp3:db "Reactome" .
  ?subPathwayXref bp3:id ?subPathwayID .
}
"""

sparql = SPARQLWrapper(endpoint_reactome)
sparql.setQuery(prefixes+query_is_a_child_of)

print("--- %s seconds ---" % (time.time() - start_time))

sparql.setReturnFormat(JSON)
results = sparql.query().convert()
displaySparqlResults(results)

sparql.setReturnFormat(CSV)
results = sparql.query().convert()
with open(f"../Results/ReactomeHomoSapiens94/ReactomeHomoSapiens94_IsAChildOf.csv", "wb") as f:
    f.write(results)

#### 2 - NextStepPathway

Defines a relation that describes the sequence of pathway steps across different pathways

In [ ]:
start_time = time.time()

query_next_step_pathway = """ 
CONSTRUCT {
  ?pathway abstraction:NextStepPathway ?nextPathway
}
WHERE {
  ?pathway rdf:type bp3:Pathway .
  ?nextPathway rdf:type bp3:Pathway .
  
  ?pathway bp3:pathwayOrder ?pathwayStep .
  ?nextPathway bp3:pathwayOrder ?nextStep .
  
  ?pathwayStep bp3:nextStep ?nextStep .
  
  FILTER (?pathway != ?nextPathway)
}
"""

sparql = SPARQLWrapper(endpoint_reactome)
sparql.setQuery(prefixes+query_next_step_pathway)
df = save_for_cytoscape(sparql, prefixes, "../Results/ReactomeHomoSapiens94/ReactomeHomoSapiens94_NextStepPathway.csv", format='csv')
preview_network_data(df)

print("--- %s seconds ---" % (time.time() - start_time))

In [13]:
# Remonter les next step pathways aux pathways parents
start_time = time.time()

query_next_step_pathway_to_parents = """ 
SELECT DISTINCT ?pathwayName ?nextName ?tag
WHERE {
  ?pathway rdf:type bp3:Pathway .
  ?nextPathway rdf:type bp3:Pathway .
  
  ?pathway bp3:pathwayOrder ?pathwayStep .
  ?nextPathway bp3:pathwayOrder ?nextStep .
  
  ?pathwayStep bp3:nextStep ?nextStep .
  
  {
    ?parentNextPathway bp3:pathwayComponent+ ?nextPathway .
    VALUES ?tag { "NextStepParentPathway" }
    BIND (?parentNextPathway AS ?next)
  }
  
  UNION
  
    {
    ?parentNextPathway bp3:pathwayComponent+ ?nextPathway .
    VALUES ?tag { "NextStepPathway" }
    BIND (?nextPathway AS ?next)
  }
  
  UNION
  
  {
    FILTER NOT EXISTS { ?parentNextPathway bp3:pathwayComponent+ ?nextPathway }
    VALUES ?tag { "NextStepPathway" }
    BIND (?nextPathway AS ?next)
  }
  
  ?pathway bp3:displayName ?pathwayName .
  ?next bp3:displayName ?nextName .
  
  FILTER (?pathway != ?nextPathway)
}
"""

sparql = SPARQLWrapper(endpoint_reactome)
sparql.setQuery(prefixes+query_next_step_pathway_to_parents)
sparql.setReturnFormat(CSV)
results = sparql.query().convert()
with open(f"../Results/ReactomeHomoSapiens94/ReactomeHomoSapiens94_NextStepPathwayToParents.csv", 'wb') as f:
    f.write(results)

print("--- %s seconds ---" % (time.time() - start_time))

16:45:31 INFO  Fuseki          :: [10] GET http://localhost:3030/reactome?query=%0APREFIX+rdf%3A+%3Chttp%3A//www.w3.org/1999/02/22-rdf-syntax-ns%23%3E%0APREFIX+rdfs%3A%3Chttp%3A//www.w3.org/2000/01/rdf-schema%23%3E%0APREFIX+owl%3A+%3Chttp%3A//www.w3.org/2002/07/owl%23%3E%0APREFIX+xsd%3A+%3Chttp%3A//www.w3.org/2001/XMLSchema%23%3E%0APREFIX+dc%3A+%3Chttp%3A//purl.org/dc/elements/1.1/%3E%0APREFIX+dcterms%3A+%3Chttp%3A//purl.org/dc/terms/%3E%0APREFIX+chebi%3A+%3Chttp%3A//purl.obolibrary.org/obo/chebi/%3E%0APREFIX+chebidb%3A+%3Chttp%3A//purl.obolibrary.org/obo/CHEBI_%3E%0APREFIX+chebirel%3A+%3Chttp%3A//purl.obolibrary.org/obo/CHEBI%23%3E%0APREFIX+oboInOwl%3A+%3Chttp%3A//www.geneontology.org/formats/oboInOwl%23%3E%0APREFIX+bp3%3A+%3Chttp%3A//www.biopax.org/release/biopax-level3.owl%23%3E%0APREFIX+reactome%3A+%3Chttp%3A//www.reactome.org/biopax/40/48887%23%3E%0APREFIX+abstraction%3A%3Chttp%3A//abstraction/%23%3E%0A+%0ASELECT+DISTINCT+%3FpathwayName+%3FnextName+%3Ftag%0AWHERE+%7B%0A++%3Fpathwa

--- 179.28926062583923 seconds ---


16:48:30 INFO  Fuseki          :: [10] 200 OK (179.277 s)


##### Process output file of NextStepPathway to pathway parents

In [4]:
next_step_to_parents = pd.read_csv("../Results/ReactomeHomoSapiens94/ReactomeHomoSapiens94_NextStepPathwayToParents.csv", header=0, sep=",")
print(next_step_to_parents.head())

                                 pathwayName  \
0                Signaling by Erythropoietin   
1                       KEAP1-NFE2L2 pathway   
2          Nuclear events mediated by NFE2L2   
3             TP53 Regulates Metabolic Genes   
4  Detoxification of Reactive Oxygen Species   

                            nextName                    tag  
0        Signaling by Erythropoietin  NextStepParentPathway  
1  Nuclear events mediated by NFE2L2  NextStepParentPathway  
2  Nuclear events mediated by NFE2L2  NextStepParentPathway  
3  Nuclear events mediated by NFE2L2  NextStepParentPathway  
4  Nuclear events mediated by NFE2L2  NextStepParentPathway  


### Concatenation of output files

In [ ]:
# concatenate output files
q1 = pd.read_csv("../Results/ReactomeHomoSapiens94/ReactomeHomoSapiens94_IsAChildOf.csv", header=0, sep=",")
q1 = q1.drop(q1.index[0]).reset_index(drop=True)
q2 = pd.read_csv("../Results/ReactomeHomoSapiens94/ReactomeHomoSapiens94_NextStepPathway.csv", header=0, sep=",")
q2 = q2.drop(q2.index[0]).reset_index(drop=True)

concat_df = pd.concat([q1, q2], ignore_index=True)

concat_df.to_csv("../Results/ReactomeHomoSapiens94/ReactomeHomoSapiens94_PathwayAbstraction.tsv", sep="\t", header=None, index=False)
print(concat_df)


In [14]:
# end process
process.kill()
time.sleep(60)